# Two-Stage Detector

A two-stage classical CV detector for X-ray security scanning:
- **Stage 1**: Object vs Background binary classification (91.9% CV accuracy)
- **Stage 2**: 6-way object class classification (82.6% CV accuracy)

Features:
- 8 proposal generation methods (multi-threshold, edge, gradient, MSER, etc.)
- 93 features per ROI (HOG, edges, texture, shape)
- Per-class confidence thresholds tuned for balanced precision/recall
- Learned size/shape constraints from training data

In [3]:
import sys

sys.path.insert(0, "../src")

from xrss.dataloader import XRayDataset
from xrss.models import TwoStageDetector
import numpy as np
import os

## 1. Load Data

In [4]:
DATA_YAML = "../xray_data/data.yaml"
PREDICTIONS_DIR = "../predictions/TwoStage_test"
os.makedirs(PREDICTIONS_DIR, exist_ok=True)

train_ds = XRayDataset(DATA_YAML, split="train")
test_ds = XRayDataset(DATA_YAML, split="test")

print(f"Train: {len(train_ds)} images")
print(f"Test: {len(test_ds)} images")

Train: 4200 images
Test: 900 images


## 2. Train Model

The model uses optimal thresholds found through experimentation:
- Stage 1 threshold: 0.45
- Per-class thresholds: {Hammer: 0.70, Knife: 0.55, Gun: 0.40, Wrench: 0.50, HandCuffs: 0.40, Bullet: 0.55}

In [5]:
detector = TwoStageDetector(
    nc=6,
    stage1_threshold=0.45,
    n_estimators=200,
)

# Train with cross-validation
detector.train(train_ds, validate=True)

Training TwoStageDetector on 4200 images...


Extracting features:   2%|▏         | 94/4200 [00:15<11:13,  6.10it/s]


KeyboardInterrupt: 

## 3. Test on Sample Images

In [ ]:
# Visualize predictions on first 8 test images
test_indices = list(range(min(8, len(test_ds))))
test_images = [test_ds[i][0] for i in test_indices]
test_labels = [test_ds[i][1] for i in test_indices]

test_preds = [detector.detect(img) for img in test_images]

print(f"Predictions on {len(test_indices)} test images:")
for i, (gt, pred) in enumerate(zip(test_labels, test_preds)):
    print(f"  Image {i}: {len(gt)} GT, {len(pred)} pred")

# Show visualizations
show_images_and_bboxes(test_images, test_preds)

## 4. Full Evaluation

In [ ]:
# Generate predictions for all test images
compute_predictions_folder(detector, test_ds, PREDICTIONS_DIR)

# Calculate mIoU
score = evaluate_score(DATA_YAML, PREDICTIONS_DIR, split="test")
print(f"\n{'=' * 50}")
print(f"Test mIoU: {score:.4f}")
print(f"{'=' * 50}")

## 5. Save Model (Optional)

In [ ]:
# Save trained model
MODEL_PATH = "../models/two_stage_detector.joblib"
os.makedirs("../models", exist_ok=True)
detector.save(MODEL_PATH)